In [ ]:
# based on: https://github.com/CamDavidsonPilon/Probabilistic-Programming-and-Bayesian-Methods-for-Hackers/blob/master/Chapter2_MorePyMC/Ch2_MorePyMC_PyMC3.ipynb

### 2.1.1 Model Context

`Model` object: any variables defined within model context (`with pm.Model() as model:`) automatically assigned to model object

In [ ]:
import pymc3 as pm

with pm.Model() as model:
    parameter = pm.Exponential("poisson_param", 1.0) #(str: variable_name, variable_value)
    data_generator = pm.Poisson("data_generator", parameter)

after model is created `with model:` is enough to continue modifying it

In [ ]:
with model:
    data_plus_one = data_generator + 1

accessing variables outside model context is OK

In [ ]:
parameter.tag.test_value

Running `with pm.Model() as model:` defines a NEW model

In [ ]:
with pm.Model() as model: # creates a new model
    theta = pm.Exponential("theta", 2.0)
    data_generator = pm.Poisson("data_generator", theta)

Now this `model` does not have a `parameter` variable anymore (old model defined in previous cells is garbage collected)

In [ ]:
with pm.Model() as ab_testing:
    p_A = pm.Uniform("P(A)", 0, 1)
    p_B = pm.Uniform("P(B)", 0, 1)

In [ ]:
print("parameter.tag.test_value =", parameter.tag.test_value)
print("data_generator.tag.test_value =", data_generator.tag.test_value)
print("data_plus_one.tag.test_value =", data_plus_one.tag.test_value)

Two things happen simultaneously:

1. The random variables get registered inside the model object
2. The Python variables parameter and data_generator get bound in the notebook's global scope

When the new model overwrites model, it only destroys the model object's internal registry. The Python variables parameter, data_generator, and data_plus_one in global scope are unaffected — they still point to the same underlying Theano tensor objects they always did.

`test_value` initial value for a variable, unchanged after sampling so that there's always a reference to how things were initialized

In [ ]:
print("parameter.tag.test_value =", parameter.tag.test_value)
print("data_generator.tag.test_value =", data_generator.tag.test_value)
print("data_plus_one.tag.test_value =", data_plus_one.tag.test_value)

can be specified at the definition of the variable via `testval=...`

In [ ]:
with pm.Model() as model:
    parameter = pm.Exponential("poisson_param", 1.0, testval=0.5)

print("\nparameter.tag.test_value =", parameter.tag.test_value)

### 2.1.2 PyMC3 Variables

In [ ]:
# with pm.Model() as example_model:
#     N = 5
#     betas = pm.Uniform("betas", 0, 1, shape=N)

In [ ]:
# with pm.Model() as example_model:
#     deterministic_variable = pm.Deterministic("deterministic variable", some_function_of_variables)

In [ ]:
# with pm.Model() as example_model:
#     lambda_1 = pm.Exponential("lambda_1", 1.0)
#     lambda_2 = pm.Exponential("lambda_2", 1.0)
#     tau = pm.DiscreteUniform("tau", lower=0, upper=10)

# new_deterministic_variable = lambda_1 + lambda_2

In [ ]:
with pm.Model() as model:
    lambda_1 = pm.Exponential("lambda_1", 1.0)
    lambda_2 = pm.Exponential("lambda_2", 1.0)
    tau = pm.DiscreteUniform("tau", lower=0, upper=10)

new_deterministic_variable = lambda_1 + lambda_2

In [ ]:
import numpy as np

n_data_points = 5  # in CH1 we had ~70 data points
idx = np.arange(n_data_points)
with model:
    lambda_ = pm.math.switch(tau >= idx, lambda_1, lambda_2)

### 2.1.3 Theano

In [ ]:
import theano.tensor as tt

with pm.Model() as theano_test:
    p1 = pm.Uniform("p1", 0, 1)
    p2 = 1 - p1
    p = tt.stack([p1, p2]) # ensures the stack is a theano variable, not NumPy variable

    assignment = pm.Categorical("assignment", p)

### 2.1.4 Including observations in the Model

In [ ]:
%matplotlib inline
from IPython.core.pylabtools import figsize
import matplotlib.pyplot as plt
import scipy.stats as stats
figsize(12.5, 4)

samples = lambda_1.random(size=20000)
plt.hist(samples, bins = 70, density = True, histtype="stepfilled")
plt.title("Prior distribution for $\lambda_1$")
plt.xlim(0, 8);

- Assume some distribution form like Poisson
- Fix distribution to observed data, PyMC will never change this data during sampling
- **Fixed variables** are therefore distinct from **free variables** which do get updated/changed during sampling

In [ ]:
data = np.array([10, 5])
with model:
    fixed_variable = pm.Poisson("fxd", 1, observed=data)
print("value: ", fixed_variable.tag.test_value)

Like how we treated data in Ch 1

In [ ]:
# We're using some fake data here
data = np.array([10, 25, 15, 20, 35])
with model:
    obs = pm.Poisson("obs", lambda_, observed=data)
print(obs.tag.test_value)

## 2.2 Modeling approaches

Generate synthetic dataset with similar assumptions made in text messaging data example

In [ ]:
tau = np.random.randint(0, 80)

alpha = 1/20
lambda_1, lambda_2 = np.random.exponential(scale=1/alpha, size=2)

data = np.r_[stats.poisson.rvs(mu=lambda_1, size=tau), stats.poisson.rvs(mu=lambda_2, size=80-tau)]

In [ ]:
tau

In [ ]:
plt.bar(np.arange(80), data, color="#348ABD")

plt.bar(tau-1, data[tau - 1], color="r", label="user behaviour changed")
plt.xlabel("Time (days)")
plt.ylabel("count of text-msgs received")
plt.title("Artificial dataset")
plt.xlim(0, 80)
plt.legend();

In [ ]:
def plot_artificial_sms_dataset():
    tau = stats.randint.rvs(0, 80)
    alpha = 1./20.
    lambda_1, lambda_2 = stats.expon.rvs(scale=1/alpha, size=2)
    data = np.r_[stats.poisson.rvs(mu=lambda_1, size=tau), stats.poisson.rvs(mu=lambda_2, size=80 - tau)]
    plt.bar(np.arange(80), data, color="#348ABD")
    plt.bar(tau - 1, data[tau-1], color="r", label="user behaviour changed")
    plt.xlim(0, 80);

figsize(12.5, 5)
plt.title("More example of artificial datasets")
for i in range(4):
    plt.subplot(4, 1, i+1)
    plot_artificial_sms_dataset()


### 2.2.1 Bayesian A/B testing

Generate fake web sales data, parameters we'd usually not know at all
- Let's say the true probability of someone visiting Site A making a purchase, $p_A$, is 5%
- Model with Bernoulli random variable

In [ ]:
#set constants
p_true = 0.05  # remember, this is unknown.
N = 10_000

# sample N Bernoulli random variables from Ber(0.05).
# each random variable has a 0.05 chance of being a 1.
# this is the data-generation step
occurrences = stats.bernoulli.rvs(p_true, size=N)

print(occurrences) # Remember: Python treats True == 1, and False == 0
print(np.sum(occurrences))

In [ ]:
# Occurrences.mean is equal to n/N.
print("What is the observed frequency in Group A? %.4f" % np.mean(occurrences))
print("Does this equal the true frequency? %s" % (np.mean(occurrences) == p_true))

Bayesian way:
- Assume uniform prior distribution for $p_A$
- Use `occurrences` as our observed variable
- Run inference

In [ ]:
with pm.Model() as model:
    p = pm.Uniform('p', lower=0, upper = 1)

In [ ]:
with model:
    obs = pm.Bernoulli('obs', p=p, observed=occurrences)
    step = pm.Metropolis()
    # trace = pm.sample(18000, step=step, chains=4, cores=1)
    trace = pm.sample(18000, step=step)
    burned_trace = trace[1000:]

In [ ]:
figsize(12.5, 4)
plt.title("Posterior distribution of $p_A$, the true effectiveness of site A")
plt.vlines(p_true, 0, 200, linestyle="--", label="true $p_A$ (unknown)", colors="red")
plt.hist(burned_trace["p"], bins=25, histtype="stepfilled", density=True)
plt.legend();

Let's introduce Site A vs Site B data

In [ ]:
figsize(12, 4)

#these two quantities are unknown to us.
true_p_A = 0.05
true_p_B = 0.04

#notice the unequal sample sizes -- no problem in Bayesian analysis.
N_A = 1500
N_B = 750

#generate some observations
observations_A = stats.bernoulli.rvs(true_p_A, size=N_A)
observations_B = stats.bernoulli.rvs(true_p_B, size=N_B)
print("Obs from Site A: ", observations_A[:30], "...")
print("Obs from Site B: ", observations_B[:30], "...")

In [ ]:
print(np.mean(observations_A))
print(np.mean(observations_B))

In [ ]:
with pm.Model() as model:
    p_A = pm.Uniform("p_A", 0, 1)
    p_B = pm.Uniform("p_B", 0, 1)

    # deterministic difference
    delta = pm.Deterministic("delta", p_A-p_B)

    # two fixed observations
    obs_A = pm.Bernoulli("obs_A", p=p_A, observed=observations_A)
    obs_B = pm.Bernoulli("obs_B", p=p_B, observed=observations_B)

    step = pm.Metropolis()
    trace = pm.sample(20_000, step=step)
    burned_trace = trace[1000:]

In [ ]:
p_A_samples = burned_trace["p_A"]
p_B_samples = burned_trace["p_B"]
delta_samples = burned_trace["delta"]

In [ ]:
figsize(12.5, 10)

#histogram of posteriors

ax = plt.subplot(311)

plt.xlim(0, .1)
plt.hist(p_A_samples, histtype='stepfilled', bins=25, alpha=0.85,
         label="posterior of $p_A$", color="#A60628", density=True)
plt.vlines(true_p_A, 0, 80, linestyle="--", label="true $p_A$ (unknown)")
plt.legend(loc="upper right")
plt.title("Posterior distributions of $p_A$, $p_B$, and delta unknowns")

ax = plt.subplot(312)

plt.xlim(0, .1)
plt.hist(p_B_samples, histtype='stepfilled', bins=25, alpha=0.85,
         label="posterior of $p_B$", color="#467821", density=True)
plt.vlines(true_p_B, 0, 80, linestyle="--", label="true $p_B$ (unknown)")
plt.legend(loc="upper right")

ax = plt.subplot(313)
plt.hist(delta_samples, histtype='stepfilled', bins=30, alpha=0.85,
         label="posterior of delta", color="#7A68A6", density=True)
plt.vlines(true_p_A - true_p_B, 0, 60, linestyle="--",
           label="true delta (unknown)")
plt.vlines(0, 0, 60, color="black", alpha=0.2)
plt.legend(loc="upper right");

Notice: 
- $N_B < N_A$ so since we collected fewer samples for site B, the posterior distribution for $p_B$ is wider, reflecting greater uncertainty
- most of the posterior for `delta` is above 0, meaning site A's better than site B in generating sales

Computing probability of site A being better or worse is straightforward

In [ ]:
delta_samples

In [ ]:
delta_samples > 0

In [ ]:
np.mean(delta_samples > 0)

In [ ]:
print("Probability site A is BETTER than site B: %.3f" % \
    np.mean(delta_samples > 0))

In [ ]:
print("Probability site A is WORSE than site B: %.3f" % \
    np.mean(delta_samples < 0))

If the above probability of site A being worse is too high, consider collecting more samples for site B.
- NOTE: this is NOT the same as Type I error even if it practically feels the same.